# 05 - Model components and baseline comparability

- **Meeting item:** Baseline fairness
- **Commit:** C07 - `feat: extract modality-agnostic model components`
- **Commit achievement:** provides neutral view encoders, concatenation fusion, and gated routing outputs.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, unknown
- **Human gate:** do all model families produce comparable outputs and routing signals?
- **Stop condition:** model names imply biology, or gate values are called explanations.

No training happens in this notebook. Weights are untrained, so the numbers below say
nothing about accuracy; they show that the interfaces line up. Model code lives in
`src/p22/models/`.

**Routing weights are signals, not explanations.** Whether a prediction depends on a routed
view is tested in C11 by intervention, not read off the gate.

In [ ]:
import dataclasses

import numpy as np
import pandas as pd
import torch

import p22
from p22.models import (
    VIEW_A,
    VIEW_B,
    VIEW_NAMES,
    BaselineMLP,
    ConcatFusionModel,
    FusionOutput,
    GatedFusionModel,
    validate_route_override,
)
from p22.testing import make_synthetic_multimodal

dataset = make_synthetic_multimodal(n_donors=6, cells_per_donor=8, seed=11)
view_a = torch.from_numpy(dataset.view_a)
view_b = torch.from_numpy(dataset.view_b)
n_classes = dataset.n_classes

torch.manual_seed(0)
models = {
    "mlp_view_a": BaselineMLP(view_a.shape[1], n_classes).eval(),
    "mlp_view_b": BaselineMLP(view_b.shape[1], n_classes).eval(),
    "concat_fusion": ConcatFusionModel(view_a.shape[1], view_b.shape[1], n_classes).eval(),
    "gated_fusion": GatedFusionModel(view_a.shape[1], view_b.shape[1], n_classes).eval(),
}

print(f"view names: {VIEW_NAMES}")
print(f"cells: {dataset.n_cells}, classes: {n_classes}")
print(f"device for all parameters: {sorted({p.device.type for m in models.values() for p in m.parameters()})}")
print(f"FusionOutput fields: {[f.name for f in dataclasses.fields(FusionOutput)]}")

## 1. Comparable outputs across families (evidence label: verified)

Every family returns logits of the same shape. The two fusion families also return routing
weights, branch embeddings, and the fused embedding.

In [ ]:
rows = []
with torch.no_grad():
    for name, model in models.items():
        if isinstance(model, BaselineMLP):
            view = view_a if name.endswith("view_a") else view_b
            logits = model(view)
            rows.append(
                {
                    "model": name,
                    "views_used": 1,
                    "logits_shape": tuple(logits.shape),
                    "routing_weights": "not applicable",
                    "has_gate": False,
                }
            )
        else:
            output = model(view_a, view_b)
            rows.append(
                {
                    "model": name,
                    "views_used": 2,
                    "logits_shape": tuple(output.logits.shape),
                    "routing_weights": tuple(output.routing_weights.shape),
                    "has_gate": model.has_gate,
                }
            )
pd.DataFrame(rows)

In [ ]:
with torch.no_grad():
    gated_output = models["gated_fusion"](view_a, view_b)
    concat_output = models["concat_fusion"](view_a, view_b)

weights = gated_output.routing_weights.numpy()
print(f"gated routing weights sum to one per cell: {np.allclose(weights.sum(axis=1), 1.0)}")
print(f"gated routing weights non-negative: {bool((weights >= 0).all())}")
print(f"gated weight on {VIEW_A}: mean {weights[:, 0].mean():.4f}, min {weights[:, 0].min():.4f}, max {weights[:, 0].max():.4f}")
print(f"concatenation reports uniform weights: {np.allclose(concat_output.routing_weights.numpy(), 0.5)}")
print("")
print("branch embedding shapes:")
for name, embedding in gated_output.branch_embeddings.items():
    print(f"  {name}: {tuple(embedding.shape)}")
print(f"fused embedding: {tuple(gated_output.fused_embedding.shape)}")
print("")
print("untrained weights: these values describe the gate at initialisation, not evidence")

## 2. Validated route overrides (evidence label: verified)

`[1, 0]` sends the fused embedding to view A alone, `[0, 1]` to view B alone, and
`[0.5, 0.5]` is the fixed uniform route used by the C11 interventions. Anything that is not
a non-negative vector summing to one is refused.

In [ ]:
gated = models["gated_fusion"]
with torch.no_grad():
    only_a = gated(view_a, view_b, route_override=[1, 0])
    only_b = gated(view_a, view_b, route_override=[0, 1])
    uniform = gated(view_a, view_b, route_override=[0.5, 0.5])

print(f"[1, 0] fused equals the {VIEW_A} branch: {torch.allclose(only_a.fused_embedding, only_a.branch_embeddings[VIEW_A])}")
print(f"[0, 1] fused equals the {VIEW_B} branch: {torch.allclose(only_b.fused_embedding, only_b.branch_embeddings[VIEW_B])}")
expected_uniform = 0.5 * uniform.branch_embeddings[VIEW_A] + 0.5 * uniform.branch_embeddings[VIEW_B]
print(f"[0.5, 0.5] fused equals the mean of both branches: {torch.allclose(uniform.fused_embedding, expected_uniform)}")
print("")
for bad in ([1, 0, 0], [0.6, 0.6], [-0.5, 1.5], [0.2, 0.2]):
    try:
        validate_route_override(bad)
    except ValueError as error:
        print(f"refused {bad}: {error}")

## 3. Branch ablation hook (evidence label: verified)

The interventions in C11 need to remove a branch, not just reweight it. Ablation replaces a
branch embedding with zeros before the gate runs, so both the weights and the fused
embedding respond.

In [ ]:
with torch.no_grad():
    baseline = gated(view_a, view_b)
    ablate_a = gated(view_a, view_b, ablate_views=[VIEW_A])
    ablate_b = gated(view_a, view_b, ablate_views=[VIEW_B])

summary = pd.DataFrame(
    [
        {
            "pass": label,
            f"mean_weight_{VIEW_A}": round(float(output.routing_weights[:, 0].mean()), 4),
            f"mean_weight_{VIEW_B}": round(float(output.routing_weights[:, 1].mean()), 4),
            "branch_a_all_zero": bool((output.branch_embeddings[VIEW_A] == 0).all()),
            "branch_b_all_zero": bool((output.branch_embeddings[VIEW_B] == 0).all()),
            "logits_changed_vs_baseline": not torch.allclose(output.logits, baseline.logits),
        }
        for label, output in (("baseline", baseline), ("ablate view_a", ablate_a), ("ablate view_b", ablate_b))
    ]
)
summary

## 4. Naming discipline (evidence label: verified)

Class, view, and parameter names are checked against a list of biological words. A model
called `MorphEncoder`, as in the legacy script, would fail this check.

In [ ]:
biological_words = ("gene", "rna", "atac", "morph", "expression", "protein", "cell_type")
names = [type(model).__name__ for model in models.values()]
names += list(VIEW_NAMES)
names += [name for name, _ in gated.named_parameters()]
hits = [(name, word) for name in names for word in biological_words if word in name.lower()]
print(f"checked {len(names)} names against {len(biological_words)} biological words")
print(f"hits: {hits or 'none'}")
print(f"class names: {sorted(set(type(model).__name__ for model in models.values()))}")

In [ ]:
assert dataset.generation["data_mode"] == p22.SYNTHETIC
assert np.allclose(gated_output.routing_weights.numpy().sum(axis=1), 1.0)
assert bool((gated_output.routing_weights >= 0).all())
assert set(gated_output.branch_embeddings) == set(VIEW_NAMES)
assert gated_output.logits.shape == (dataset.n_cells, n_classes)
assert concat_output.logits.shape == (dataset.n_cells, n_classes)
assert models["concat_fusion"].has_gate is False and gated.has_gate is True
assert torch.allclose(only_a.fused_embedding, only_a.branch_embeddings[VIEW_A])
assert torch.allclose(only_b.fused_embedding, only_b.branch_embeddings[VIEW_B])
assert bool((ablate_a.branch_embeddings[VIEW_A] == 0).all())
assert hits == []
assert not hasattr(gated, "fit")
print("all C07 gate checks passed")

## 5. Unknown (evidence label: unknown)

- Whether gated fusion helps at all. Untrained forward passes cannot answer that; C10 measures it across five seeds.
- Whether the gate's weights track anything the prediction depends on. C11 tests that by intervention.
- Which encoder widths, embedding size, or dropout suit a real dataset. Current values are defaults for a small synthetic world, not tuned choices.

## Gate

**PASS**

- **Observed:** four model families return logits of shape (48, 3) on the same inputs; gated fusion returns routing weights of shape (48, 2) that are non-negative and sum to one per cell; concatenation fusion has no gate and reports uniform 0.5 weights so reporting code can treat families alike; `FusionOutput` carries exactly `logits`, `routing_weights`, `branch_embeddings`, `fused_embedding`; overrides `[1, 0]`, `[0, 1]`, and `[0.5, 0.5]` behave as defined and malformed overrides are refused; branch ablation zeros a branch and changes both weights and logits; no class, view, or parameter name contains a biological word; all parameters are on CPU; no model exposes a training entry point.
- **Remains unknown:** whether gated fusion helps, whether gate weights track anything the prediction depends on, and what architecture sizes suit a real dataset.
- **Next decision owner:** researcher continues with C08 (metrics and donor uncertainty).